In [20]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langgraph.checkpoint.memory import InMemorySaver
from ipywidgets import FileUpload
from IPython.display import display
import base64
from dotenv import load_dotenv

load_dotenv()

True

In [21]:
tavily_client = TavilyClient()

In [22]:
@tool
def web_search(query: str) -> Dict[str, Any]:
    """
    Perform a web search using the Tavily API.

    Args:
        query (str): The search query.

    Returns:
        Dict[str, Any]: The search results.
    """
    return tavily_client.search(query)

In [28]:
SYSTEM_PROMPT = """"
    You are an Indian home chef that provides food recommendations based on the groceries available in the user's fridge. 
    
    You will be provided with the image of the fridge and you will need to analyze the image to determine what groceries are available. 
    
    You will then provide a list of possible dishes that can be made with the available groceries by using the available tools to search the web when necessary, along with a brief description of each dish. 
    
    You will also provide a recipe for each dish, including the ingredients and detailed instructions for preparation. Your responses should be concise and informative, providing the user with clear and actionable recommendations.
"""

In [24]:
agent = create_agent(
    model="anthropic:claude-haiku-4-5",
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    tools=[web_search],
)

In [25]:
uploader = FileUpload(accept=".jpeg", multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpeg', description='Upload')

In [ ]:
image_base64 = base64.b64encode(bytes(uploader.value[0]["content"])).decode("utf-8")

multimodal_question = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "What should I eat today based on the groceries available in my fridge?",
        },
        {"type": "image", "base64": image_base64, "mime_type": "image/jpeg"},
    ]
)

response = agent.invoke(
    {
        "messages": [multimodal_question],
    },
    # Always set the thread_id to "1" to ensure that the conversation is always in the same thread, 
    # to ensure that the agent has access to the previous messages in the conversation and to ask follow up questions.
    config={"configurable": {"thread_id": "1"}},  # type: ignore
)

In [27]:
print(response["messages"][-1].content)

Looking at your fridge, I can see some wonderful fresh vegetables! Let me analyze what you have:

**Available Groceries:**
- Mixed greens/lettuce
- Cauliflower
- Tomatoes (in the red mesh bag)
- Broccoli or similar greens
- Bell peppers (red and green)
- Cucumber or similar green vegetables
- Whole wheat bread

Based on these fresh vegetables, here are some delicious Indian dishes you can prepare:

## **1. Sabzi ka Paratha (Vegetable Stuffed Flatbread)**
A wholesome Indian flatbread stuffed with mixed greens and vegetables, perfect for breakfast or lunch. You can use your bread as a base or make parathas from scratch if you have flour.

**Quick Recipe:**
- Knead dough (if making fresh) or use your bread
- Mix finely chopped greens, cauliflower florets, bell peppers, and spices
- Stuff, roll, and cook on a griddle with a bit of oil until golden
- Serve with yogurt or pickle

---

## **2. Vegetable Stir-Fry (Sabzi Bhujia)**
A quick, nutritious dish highlighting all your fresh vegetables.